# TSFS12 Hand-in exercise 4: Collaborative Control 5.1

In [152]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [153]:
%matplotlib qt

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [154]:
def double_integrator(t, x, u, model_params):
    '''
    System-model functions
     
     Input arguments are:
     t - time 
     x - agent state
     u - the control input u, 
     model_params - structure with parameters of the model '''

    velocity = x[2:4]
    wind_acc = model_params["wind_force"] / model_params["m"]
    return np.concatenate((velocity, u + wind_acc))

def controller_p_v(y, y_ref, ctrl_params):
    ''' 
    Control function   

    Output: Control signal 
    '''
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    p, v = y[:2], y[2:]
    p_ref, v_ref = y_ref[:2], y_ref[2:]

    return Kp * (p_ref - p) + Kv * (v_ref - v)

Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [155]:
n = 4  # (px, py, vx, vy)
G = [
    (0,),  # Agent 0
    (1,),  # Agent 1
    (2,),  # Agent 2
    (3,),  # Agent 3
]

formation_references = [
    # Circle: p = (cos(2t), sin(2t))
    lambda t: np.array([
        np.cos(2 * t), np.sin(2 * t),
        -2 * np.sin(2 * t), 2 * np.cos(2 * t),
    ]),

    # Clockwise circle: p = (2 cos(-t), 2 sin(-t))
    lambda t: np.array([
        2 * np.cos(-t), 2 * np.sin(-t),
        2 * np.sin(-t), -2 * np.cos(-t),
    ]),

    lambda t: [3, 2, 0, 0], # Fixed position
    lambda t: [5, 4, 0, 0], # Fixed position
]

Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

In [156]:
modelparam = {"m": 1, "wind_force": np.array([4.0, 0.0])}
ctrl_params_1 = {"kp": 4, "kv": 4}

agents = [
    CreateAgent(double_integrator, modelparam, controller_p_v, ctrl_params_1),
    CreateAgent(double_integrator, modelparam, controller_p_v, ctrl_params_1),
    CreateAgent(double_integrator, modelparam, controller_p_v, ctrl_params_1),
    CreateAgent(double_integrator, modelparam, controller_p_v, ctrl_params_1),
]

formation = AgentFormation(agents, G, formation_references, n)

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [157]:
x0 = np.array([
    1, 0, 0, 0,  # Agent 0
    2, 0, 0, 0,  # Agent 1
    3, 0, 0, 0,  # Agent 2
    4, 0, 0, 0,  # Agent 3
])

Simulate the formation

In [158]:
t = np.arange(0, 10, 0.05) # Time vector for the simulation
x = odeint(formation.ode, x0, t)

Animate the results

In [159]:
fig, ax = plt.subplots(num=20, clear=True, figsize=(8, 6))
ax.axis([-3, 7, -4, 5])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Exercise 5.1: Agent trajectories")

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
params_text = ax.text(
    0.02, 0.02,
    f"Kp = {ctrl_params_1['kp']}, Kv = {ctrl_params_1['kv']}\n"
    f"m = {modelparam['m']}, wind force = {modelparam['wind_force']}",
    transform=ax.transAxes,
    ha="left", va="bottom",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
)

def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x[: i + 1, n * idx]
        py = x[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text, params_text)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x.shape[0], blit=True, repeat=False
)
